How do the adapters compare when viewed across all four benchmarks simultaneously?

This page uses three complementary views — without collapsing multi-dimensional performance into a single number that would paper over meaningful tradeoffs.

**Adapters:** SQLite FTS5, LanceDB, ChromaDB, Tantivy, Qdrant  
**Benchmarks:** Code Finding, Doc Search, Episodic Memory, Skill Search  
**Primary metric:** nDCG@10 (higher is better)

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path().resolve()
for _p in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (_p / 'results').exists(): ROOT = _p; break

BENCHMARKS = ['code-finding', 'doc-search', 'episodic-memory', 'skill-search']
BENCH_LABELS = {
    'code-finding': 'Code\nFinding',
    'doc-search': 'Doc\nSearch',
    'episodic-memory': 'Episodic\nMemory',
    'skill-search': 'Skill\nSearch',
}
ADAPTERS = ['sqlite', 'lancedb', 'chromadb', 'tantivy', 'qdrant']
ADAPTER_LABELS = {
    'sqlite': 'SQLite FTS5', 'lancedb': 'LanceDB',
    'chromadb': 'ChromaDB', 'tantivy': 'Tantivy', 'qdrant': 'Qdrant',
}
ADAPTER_COLORS = {
    'sqlite': '#f28e2b', 'lancedb': '#4e79a7', 'chromadb': '#59a14f',
    'tantivy': '#e15759', 'qdrant': '#edc948',
}

rows = []
for b in BENCHMARKS:
    for a in ADAPTERS:
        d = ROOT / 'results' / b / a
        files = sorted(d.glob('*.json')) if d.exists() else []
        if files:
            r = json.loads(files[-1].read_text())
            rows.append({
                'benchmark': b, 'adapter': a,
                'ndcg_at_10': r.get('ndcg_at_10', np.nan),
                'latency_p50_ms': r.get('latency_p50_ms', np.nan),
            })

df = pd.DataFrame(rows)
pivot = df.pivot(index='adapter', columns='benchmark', values='ndcg_at_10')
print(pivot.round(3).to_string())

## Overlaid Radar: All Adapters

Each adapter's profile across the four benchmarks. A "full" shape means strong performance everywhere; gaps indicate benchmark-specific weaknesses.

In [ ]:
bench_order = BENCHMARKS
N = len(bench_order)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()

def radar_chart(ax, values, color, label, alpha_fill=0.12):
    values_c = values + values[:1]
    angles_c = angles + angles[:1]
    ax.plot(angles_c, values_c, color=color, linewidth=1.8, label=label)
    ax.fill(angles_c, values_c, color=color, alpha=alpha_fill)

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#fafafa')
ax.set_facecolor('#fafafa')

for a in ADAPTERS:
    if a not in pivot.index: continue
    vals = [float(pivot.loc[a, b]) if b in pivot.columns and not pd.isna(pivot.loc[a, b]) else 0.0
            for b in bench_order]
    radar_chart(ax, vals, ADAPTER_COLORS[a], ADAPTER_LABELS[a])

ax.set_xticks(angles)
ax.set_xticklabels([BENCH_LABELS[b] for b in bench_order], fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8, color='#888')
ax.grid(color='#ccc', linestyle=':', linewidth=0.8)
ax.spines['polar'].set_visible(False)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)
ax.set_title('nDCG@10 by adapter and benchmark', fontsize=11, pad=20)
plt.tight_layout()
plt.show()

## Per-Adapter Profiles

The same data as individual profiles — useful for understanding each adapter's strengths in isolation.

In [ ]:
adapters_present = [a for a in ADAPTERS if a in pivot.index]
ncols = 3
nrows = (len(adapters_present) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows),
                          subplot_kw=dict(polar=True))
fig.patch.set_facecolor('#fafafa')
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, a in enumerate(adapters_present):
    ax = axes_flat[i]
    ax.set_facecolor('#fafafa')
    vals = [float(pivot.loc[a, b]) if b in pivot.columns and not pd.isna(pivot.loc[a, b]) else 0.0
            for b in bench_order]
    radar_chart(ax, vals, ADAPTER_COLORS[a], ADAPTER_LABELS[a], alpha_fill=0.25)
    ax.set_xticks(angles)
    ax.set_xticklabels([BENCH_LABELS[b] for b in bench_order], fontsize=9)
    ax.set_ylim(0, 1.0)
    ax.set_yticks([0.5, 1.0])
    ax.set_yticklabels(['0.5', '1.0'], fontsize=7, color='#888')
    ax.grid(color='#ccc', linestyle=':', linewidth=0.8)
    ax.spines['polar'].set_visible(False)
    ax.set_title(ADAPTER_LABELS[a], fontsize=10, pad=12,
                 color=ADAPTER_COLORS[a], fontweight='bold')

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Per-adapter nDCG@10 profiles', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Cross-Benchmark Heatmap

Color intensity = nDCG@10. Scan across a row to see one adapter's performance; scan down a column to compare adapters on one benchmark.

In [ ]:
adapters_present = [a for a in ADAPTERS if a in pivot.index]
heat_data = pivot.loc[adapters_present, bench_order].values.astype(float)

fig, ax = plt.subplots(figsize=(7, 3.5))
fig.patch.set_facecolor('#fafafa')
ax.set_facecolor('#fafafa')
im = ax.imshow(heat_data, aspect='auto', cmap='YlGn', vmin=0, vmax=1)

ax.set_xticks(range(len(bench_order)))
ax.set_xticklabels([BENCH_LABELS[b].replace('\n', ' ') for b in bench_order], fontsize=10)
ax.set_yticks(range(len(adapters_present)))
ax.set_yticklabels([ADAPTER_LABELS[a] for a in adapters_present], fontsize=10)

for i, a in enumerate(adapters_present):
    for j, b in enumerate(bench_order):
        v = heat_data[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.3f}', ha='center', va='center',
                    fontsize=9, color='black' if v < 0.7 else 'white', fontweight='bold')

plt.colorbar(im, ax=ax, label='nDCG@10', shrink=0.8)
ax.set_title('Adapter \u00d7 Benchmark nDCG@10', fontsize=11)
plt.tight_layout()
plt.show()

## Speed-Accuracy Trade-off

Where does each adapter sit on the Pareto frontier? Latency is query p50 (ms), accuracy is mean nDCG@10 across all four benchmarks. Lower-left is fast-but-weak; upper-right is accurate-but-slow.

In [ ]:
lat_df = df.groupby('adapter').agg(
    mean_ndcg=('ndcg_at_10', 'mean'),
    mean_latency=('latency_p50_ms', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(6, 4))
fig.patch.set_facecolor('#fafafa')
ax.set_facecolor('#fafafa')
for s in ax.spines.values(): s.set_visible(False)

for _, row in lat_df.iterrows():
    a = row['adapter']
    if pd.isna(row['mean_ndcg']) or pd.isna(row['mean_latency']): continue
    ax.scatter(row['mean_latency'], row['mean_ndcg'],
               color=ADAPTER_COLORS.get(a, '#aaa'), s=120, zorder=5)
    ax.annotate(ADAPTER_LABELS.get(a, a),
                (row['mean_latency'], row['mean_ndcg']),
                xytext=(6, 3), textcoords='offset points', fontsize=9)

ax.set_xscale('log')
ax.set_xlabel('Query latency p50 (ms, log scale)', fontsize=10)
ax.set_ylabel('Mean nDCG@10 (all benchmarks)', fontsize=10)
ax.set_title('Speed-accuracy trade-off', fontsize=11)
ax.yaxis.grid(True, linestyle=':', alpha=0.6)
ax.xaxis.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

## Analysis

**No single winner.** Adapter choice depends on query type:

- **Dense-first tasks** (Skill Search, Code Finding): LanceDB, ChromaDB, and Qdrant lead because user vocabulary diverges from document vocabulary — semantic similarity bridges the gap BM25 cannot.
- **Keyword-rich tasks** (Episodic Memory, Doc Search): SQLite FTS5 and Tantivy match or beat dense adapters because the corpus uses domain-specific terminology that appears verbatim in queries.

**Latency** varies by an order of magnitude. SQLite FTS5 and Tantivy are consistently the fastest. Dense adapters pay a fixed overhead for embedding computation but scale predictably.

**Qdrant** performs competitively across all tasks — strong on dense-friendly benchmarks (0.585 on Code Finding, 0.925 on Skill Search), and weaker but not outlying on BM25-friendly tasks.

**The radar chart tells the full story.** An adapter with a balanced, full-polygon shape is a safe default for general-purpose agent memory. An adapter with spikes in specific quadrants is a specialist. SQLite FTS5 and Tantivy are BM25 specialists; ChromaDB, LanceDB, and Qdrant are semantic specialists.